# 02 — Experimental Strategies

Four strategies with real academic or practitioner backing. Each has a plausible **economic rationale** for why it might work — not just a pattern that fits historical data.

| # | Strategy | Type | Origin |
|---|---|---|---|
| 1 | Donchian Channel Breakout | Trend-following | Turtle Trading system (1983) |
| 2 | Time Series Momentum | Trend-following | Moskowitz, Ooi, Pedersen (2012) |
| 3 | RSI + Trend Filter | Mean reversion + regime | Combines two classics |
| 4 | Dual Momentum | Cross-asset momentum | Gary Antonacci (2014) |

> **Important:** A good backtest is necessary but not sufficient. All strategies here are tested on SPY 2010–2024, which includes a prolonged bull market. Be skeptical of your own results.

---

In [ ]:
import sys
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings

from DrakonixBacktester import Backtester, Strategy
from DrakonixBacktester.strategies import BuyAndHold
from DrakonixBacktester import metrics

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (13, 6)

INITIAL_CAPITAL = 10_000
START, END = '2010-01-01', '2025-01-01'

In [ ]:
# Download data for all strategies
# SPY  = S&P 500 (our primary test asset)
# AGG  = US Aggregate Bond ETF (used in Dual Momentum)
# BIL  = 1-3 Month T-Bill ETF (cash proxy in Dual Momentum)
raw = yf.download(['SPY', 'AGG', 'BIL'], start=START, end=END, auto_adjust=True)
closes = raw['Close'].dropna()

spy   = closes['SPY']
agg   = closes['AGG']
bil   = closes['BIL']

# Buy-and-hold baseline (computed once, reused for all plots)
bh_result = Backtester(spy, BuyAndHold(), INITIAL_CAPITAL).run()

print(f'SPY: {len(spy)} bars ({spy.index[0].date()} – {spy.index[-1].date()})')

---

## Strategy 1 — Donchian Channel Breakout (Turtle Trading)

**Origin:** Richard Dennis and Bill Eckhardt's famous Turtle Trading experiment (1983). Dennis bet that trading could be taught like any other skill — he trained a group of novices (the "turtles") on this system and they made ~$100M.

**Logic:**
- **Buy** when today's close breaks above the highest close of the last `entry` days
- **Sell** when today's close falls below the lowest close of the last `exit` days

**Why it might work:** Markets trend. When price breaks out to a new N-day high, it's evidence that buyers are in control and the trend may continue. This is the most direct implementation of "cut your losses, let your winners run."

**Why it fails sometimes:** In sideways markets, the strategy whipsaws — it keeps buying new highs and selling new lows, accumulating small losses.

In [ ]:
class DonchianBreakout(Strategy):
    """
    Donchian Channel Breakout (Turtle Trading system).

    Entry: price breaks above the highest close of the last `entry` bars.
    Exit:  price falls below the lowest close of the last `exit` bars.

    The asymmetric entry/exit (longer entry, shorter exit) is intentional:
    it lets winners run longer while cutting losers quickly.
    """

    def __init__(self, entry: int = 20, exit_: int = 10):
        self.entry = entry
        self.exit_ = exit_
        self._in_trade = False

    def generate_signal(self, prices: pd.Series) -> int:
        if len(prices) < self.entry + 1:
            return 0

        price = float(prices.iloc[-1])
        # Exclude today from the channel calculation (avoid look-ahead)
        entry_high = float(prices.iloc[-self.entry - 1:-1].max())
        exit_low   = float(prices.iloc[-self.exit_ - 1:-1].min())

        if not self._in_trade and price > entry_high:
            self._in_trade = True
            return 1

        if self._in_trade and price < exit_low:
            self._in_trade = False
            return -1

        return 0

    def reset(self):
        self._in_trade = False


dc_result = Backtester(spy, DonchianBreakout(entry=20, exit_=10), INITIAL_CAPITAL).run()

print(f'Trades: {len(dc_result.trades)}')
print(dc_result.summary())
dc_result.plot('Donchian Breakout (20/10) — SPY 2010–2025', benchmark=bh_result.equity)

---

## Strategy 2 — Time Series Momentum (TSMOM)

**Origin:** Moskowitz, Ooi & Pedersen, *"Time Series Momentum"* (Journal of Financial Economics, 2012). One of the most cited quant finance papers. Tested across 58 liquid instruments — futures, FX, commodities, bonds.

**Logic:** If an asset's return over the past 12 months is positive, hold it long. If negative, go flat (or short, in the original paper).

**Why it might work:** Momentum has two potential explanations:
1. **Behavioral:** Investors underreact to news initially (anchoring), then overreact later (herding) — creating persistent trends
2. **Risk-based:** Trending assets may carry a risk premium that compensates holders for bearing trend risk

**Vol scaling:** The original paper also scales position size by `target_vol / realized_vol` so that each position contributes equal risk. Our backtester is all-in or flat, so we implement the binary version here — but the vol scaling is the most important improvement you can add.

In [ ]:
class TimeSeriesMomentum(Strategy):
    """
    Binary time series momentum.

    Go long if 12-month return is positive, flat if negative.

    The original Moskowitz et al. paper also scales position size by
    target_vol / realized_vol — this implementation skips that (all-in
    or flat) but captures the directional signal.

    Args:
        lookback: return measurement window in trading days (default 252 = 1 year)
        skip:     skip the most recent `skip` days before measuring the signal.
                  The original paper skips 1 month to avoid short-term reversal.
    """

    def __init__(self, lookback: int = 252, skip: int = 21):
        self.lookback = lookback
        self.skip = skip

    def generate_signal(self, prices: pd.Series) -> int:
        required = self.lookback + self.skip + 1
        if len(prices) < required:
            return 0

        # Return from lookback+skip days ago to skip days ago
        # (skip the most recent month to avoid short-term reversal contamination)
        price_now  = float(prices.iloc[-self.skip - 1])
        price_then = float(prices.iloc[-self.lookback - self.skip - 1])
        ret_12m = (price_now - price_then) / price_then

        return 1 if ret_12m > 0 else -1


tsmom_result = Backtester(spy, TimeSeriesMomentum(lookback=252, skip=21), INITIAL_CAPITAL).run()

print(f'Trades: {len(tsmom_result.trades)}')
print(tsmom_result.summary())
tsmom_result.plot('Time Series Momentum (12m, skip 1m) — SPY 2010–2025', benchmark=bh_result.equity)

---

## Strategy 3 — RSI Mean Reversion + Trend Filter

**Problem with pure mean reversion:** A stock can be "oversold" by RSI and keep falling for months (think: a company going bankrupt, or a market in freefall). RSI oversold in a downtrend is a trap.

**The fix:** Only take mean-reversion entries when the **broader trend is up** — i.e., price is above its 200-day SMA. This is a simple **regime filter**.

**Logic:**
- BUY when: RSI < 30 **AND** price > 200-day SMA
- SELL when: RSI crosses back above 50
- Stay flat (don't buy dips) when price is below 200-day SMA

**Why it might work:** You're combining two signals with complementary failure modes. The trend filter prevents you from buying into sustained downtrends. The RSI entry times your entry within an uptrend to get a better price.

In [ ]:
class RSIWithTrendFilter(Strategy):
    """
    RSI mean reversion, only active in an uptrend.

    Entry:  RSI < oversold_threshold AND price > 200-day SMA
    Exit:   RSI >= 50 (mean reversion complete)
    Filter: if price < 200-day SMA, no new entries (but will exit open positions)

    Args:
        rsi_window:  RSI lookback period (default 14)
        oversold:    RSI buy threshold (default 30)
        trend_sma:   trend filter SMA period (default 200)
    """

    def __init__(self, rsi_window: int = 14, oversold: float = 30.0, trend_sma: int = 200):
        self.rsi_window = rsi_window
        self.oversold = oversold
        self.trend_sma = trend_sma
        self._in_trade = False

    def _rsi(self, prices: pd.Series) -> float:
        delta = prices.diff().dropna()
        gain = delta.clip(lower=0).rolling(self.rsi_window).mean()
        loss = (-delta.clip(upper=0)).rolling(self.rsi_window).mean()
        rs = gain / loss.replace(0, float('inf'))
        return float(100 - (100 / (1 + rs.iloc[-1])))

    def generate_signal(self, prices: pd.Series) -> int:
        if len(prices) < self.trend_sma + self.rsi_window + 1:
            return 0

        price     = float(prices.iloc[-1])
        sma200    = float(prices.iloc[-self.trend_sma:].mean())
        rsi       = self._rsi(prices)
        in_uptrend = price > sma200

        if not self._in_trade and in_uptrend and rsi < self.oversold:
            self._in_trade = True
            return 1

        if self._in_trade and rsi >= 50:
            self._in_trade = False
            return -1

        return 0

    def reset(self):
        self._in_trade = False


rsi_tf_result = Backtester(spy, RSIWithTrendFilter(), INITIAL_CAPITAL).run()

print(f'Trades: {len(rsi_tf_result.trades)}')
print(rsi_tf_result.summary())
rsi_tf_result.plot('RSI + Trend Filter — SPY 2010–2025', benchmark=bh_result.equity)

---

## Strategy 4 — Dual Momentum (Antonacci)

**Origin:** Gary Antonacci, *"Dual Momentum Investing"* (2014). Combines two momentum concepts:

1. **Relative momentum (cross-sectional):** Which asset is stronger — equities (SPY) or bonds (AGG)?
2. **Absolute momentum (time series):** Is the winner also positive over the last 12 months? If not, hold cash.

**Monthly rebalancing logic:**
- Compare 12-month return of SPY vs AGG
- If SPY wins AND SPY return > 0 → hold SPY
- If AGG wins AND AGG return > 0 → hold AGG
- If the winner's 12m return is negative → hold cash (BIL)

**Why it might work:** It rotates into the strongest asset class, and the absolute momentum filter keeps you out of bear markets entirely — avoiding the worst crashes.

**Note:** This is a multi-asset strategy, so we implement it with a vectorized backtest rather than the DrakonixBacktester framework (which works on a single price series).

In [ ]:
def dual_momentum_backtest(
    spy: pd.Series,
    agg: pd.Series,
    bil: pd.Series,
    lookback_months: int = 12,
    initial_capital: float = 10_000,
) -> pd.Series:
    """
    Vectorized Dual Momentum backtest.

    Rebalances monthly. Signal is computed on the last trading day of each
    month and applied at the close of the first trading day of the next month.

    Returns:
        pd.Series: daily equity curve
    """
    # Align all series to common dates
    prices = pd.DataFrame({'SPY': spy, 'AGG': agg, 'BIL': bil}).dropna()

    # Compute monthly returns (approximate lookback as ~21 trading days per month)
    lookback_days = lookback_months * 21
    ret = prices / prices.shift(lookback_days) - 1

    # Resample to month-end to generate monthly signals
    monthly_ret = ret.resample('ME').last()

    # Determine holding for next month
    def pick_asset(row):
        spy_ret, agg_ret, bil_ret = row['SPY'], row['AGG'], row['BIL']
        if pd.isna(spy_ret) or pd.isna(agg_ret):
            return 'BIL'
        if spy_ret >= agg_ret and spy_ret > 0:
            return 'SPY'
        elif agg_ret > spy_ret and agg_ret > 0:
            return 'AGG'
        else:
            return 'BIL'

    monthly_holding = monthly_ret.apply(pick_asset, axis=1)
    # Shift by 1 month: signal from end of month M applies during month M+1
    monthly_holding = monthly_holding.shift(1)

    # Forward-fill to daily frequency
    daily_holding = monthly_holding.reindex(prices.index, method='ffill').dropna()
    prices = prices.loc[daily_holding.index]

    # Compute daily returns of the held asset
    daily_rets = prices.pct_change().dropna()
    daily_holding = daily_holding.reindex(daily_rets.index, method='ffill').dropna()

    strategy_rets = pd.Series(
        [daily_rets.loc[d, h] for d, h in daily_holding.items() if d in daily_rets.index],
        index=[d for d in daily_holding.index if d in daily_rets.index],
    )

    equity = initial_capital * (1 + strategy_rets).cumprod()
    return equity


dm_equity = dual_momentum_backtest(spy, agg, bil, initial_capital=INITIAL_CAPITAL)
dm_equity = dm_equity.dropna()

print('Dual Momentum (12-month, monthly rebalance)')
print(metrics.summary(dm_equity))

In [ ]:
# Plot Dual Momentum equity curve with drawdown
import matplotlib.gridspec as gridspec

bh_aligned = bh_result.equity / bh_result.equity.iloc[0] * INITIAL_CAPITAL

fig = plt.figure(figsize=(13, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax1.plot(dm_equity, color='steelblue', linewidth=1.5, label='Dual Momentum')
ax1.plot(bh_aligned.reindex(dm_equity.index, method='nearest'),
         color='gray', linewidth=1, linestyle='--', alpha=0.7, label='Buy & Hold SPY')
ax1.set_title('Dual Momentum (SPY/AGG/Cash) — 2010–2025')
ax1.set_ylabel('Portfolio Value ($)')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend()
plt.setp(ax1.get_xticklabels(), visible=False)

rolling_max = dm_equity.cummax()
drawdown = (dm_equity - rolling_max) / rolling_max
ax2.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.4)
ax2.set_ylabel('Drawdown')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

---

## Comparison — All Strategies

In [ ]:
# Align Dual Momentum to the other series' start date
common_start = max(
    bh_result.equity.index[0],
    dc_result.equity.index[0],
    tsmom_result.equity.index[0],
    rsi_tf_result.equity.index[0],
    dm_equity.index[0],
)

def norm(s):
    s = s[s.index >= common_start]
    return s / s.iloc[0] * INITIAL_CAPITAL

comparison = pd.DataFrame({
    'Buy & Hold':          metrics.summary(norm(bh_result.equity)),
    'Donchian (20/10)':    metrics.summary(norm(dc_result.equity)),
    'TSMOM (12m)':         metrics.summary(norm(tsmom_result.equity)),
    'RSI + Trend Filter':  metrics.summary(norm(rsi_tf_result.equity)),
    'Dual Momentum':       metrics.summary(norm(dm_equity)),
})
print(comparison.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

styles = [
    (bh_result.equity,     'Buy & Hold',         'gray',      '--', 1.2),
    (dc_result.equity,     'Donchian (20/10)',    'steelblue', '-',  1.5),
    (tsmom_result.equity,  'TSMOM (12m)',         'green',     '-',  1.5),
    (rsi_tf_result.equity, 'RSI + Trend Filter',  'darkorange','-',  1.5),
    (dm_equity,            'Dual Momentum',       'purple',    '-',  1.5),
]

for series, label, color, ls, lw in styles:
    n = norm(series)
    ax.plot(n, label=label, color=color, linestyle=ls, linewidth=lw)

ax.set_title('All Strategies — Normalized to $10,000')
ax.set_ylabel('Portfolio Value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

---

## What to Notice

**Drawdown is the real test.** A strategy that returns 15%/year but has a -50% drawdown is nearly unrunnable — most people will panic-sell at the bottom. Look at *when* each strategy's worst drawdowns occur.

**Sharpe > Total Return.** A strategy with a lower total return but higher Sharpe ratio is often *better* in practice — it's more consistent, easier to size up, and psychologically easier to stick with.

**This is in-sample.** All strategies were tested on data that exists right now. They may have been implicitly designed around known market regimes (2010–2025 was mostly a bull market). Out-of-sample testing on different assets or time periods is the real test.

**Next steps to make these more rigorous:**
1. Walk-forward validation (re-fit parameters on rolling windows)
2. Test on multiple assets, not just SPY
3. Stress-test on pre-2010 data (dot-com crash, 2008 crisis)
4. Add realistic transaction costs and slippage
5. Deflated Sharpe Ratio (corrects for multiple comparisons)